In [ ]:
import pandas as pd
from sqlalchemy import create_engine

df = pd.read_csv('EmendasParlamentares.csv', sep = ';', encoding='latin-1')
print(f"O arquivo possui {len(df)} linhas e {len(df.columns)} colunas.")
df.head()

In [ ]:
df ['Valor Empenhado'] = df ['Valor Empenhado'].astype(str).str.replace(',','', regex=False).str.replace('.','.', regex=False)
df ['Valor Empenhado'] = pd.to_numeric(df ['Valor Empenhado'], errors='coerce').fillna(0)

df ['Nome do Autor da Emenda'] = df ['Nome do Autor da Emenda'].str.upper().str.strip()
df ['Nome Função'] = df ['Nome Função'].str.upper().str.strip()

import numpy as np
df = df.replace('Sem Informação', np.nan)

colunas_selecionadas = ['Ano da Emenda', 'Tipo de Emenda', 'Nome do Autor da Emenda', 'Município', 'UF', 'Nome Função', 'Valor Empenhado', ]
df_final = df[colunas_selecionadas].copy()

print("Limpeza Concluida com Sucesso!")
df_final.head()

In [ ]:
engine = create_engine('sqlite:///transparencia_emendas.db')
df_final.to_sql('emendas_parlamentares', con=engine, if_exists='replace', index=False)

print("Dados inseridos no banco de dados com sucesso!")

In [ ]:
termos_para_remover = ['BANCADA', 'COM.', 'RELATOR', 'SEM INFORMAÇÃO']
filtro = ~df_final['Nome do Autor da Emenda'].str.contains('|'.join(termos_para_remover), case=False, na=False)
df_parlamentares = df_final[filtro]

top_autores = df_parlamentares.groupby('Nome do Autor da Emenda')['Valor Empenhado'].sum().sort_values(ascending=False)
print("Analise Concluida com Sucesso!\n")
for nome, valor in top_autores.head(10).items():
    print(f"{nome}: R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

In [ ]:
termos_para_remover_uf = ['SEM INFORMAÇÃO', 'Múltiplo']
filtro_uf = ~df_final['UF'].str.contains('|'.join(termos_para_remover_uf), case=False, na=False)
df_uf = df_final[filtro_uf]

verba_por_uf = df_uf.groupby('UF')['Valor Empenhado'].sum().sort_values(ascending=False)
print("Analise Concluida com Sucesso!\n")
for nome, valor in verba_por_uf.head(10).items():
    print(f"{nome}: R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.'))

In [ ]:
df_parlamentares.to_csv('EmendasParlamentares_Limpo.csv', index=False, encoding='UTF-8')